In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, Model
from tensorflow.keras.utils import Sequence
from sklearn.model_selection import train_test_split

print("Imports done!")

In [ ]:
BASE_PATH = '/kaggle/input/datasets/organizations/nih-chest-xrays/data'

DISEASES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion',
    'Infiltration', 'Mass', 'Nodule', 'Pneumonia',
    'Pneumothorax', 'Consolidation', 'Edema',
    'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia'
]

df = pd.read_csv(f'{BASE_PATH}/Data_Entry_2017.csv')

df = df[df['Finding Labels'] != 'No Finding']

# 20% subset of each disease
df = df.groupby('Finding Labels').apply(
    lambda x: x.sample(frac=0.2, random_state=42)
).reset_index(drop=True)

print("Subset ready!")
print("Total images:", len(df))

for disease in DISEASES:
    count = df['Finding Labels'].str.contains(disease).sum()
    print(f"{disease}: {count} images")

# Encoding Labels
def encode_labels(label_string):
    labels = label_string.split('|')
    return [1 if disease in labels else 0 for disease in DISEASES]

df['encoded_labels'] = df['Finding Labels'].apply(encode_labels)

# Image path function
def get_image_path(image_name):
    folders = ['images_001', 'images_002', 'images_003',
               'images_004', 'images_005', 'images_006',
               'images_007', 'images_008', 'images_009',
               'images_010', 'images_011', 'images_012']
    for folder in folders:
        path = f'{BASE_PATH}/{folder}/images/{image_name}'
        if os.path.exists(path):
            return path
    return None

# Train test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print("\n Split done!")
print("Train:", len(train_df), "| Test:", len(test_df))

In [ ]:
class XRayGenerator(Sequence):
    def __init__(self, dataframe, batch_size=128, shuffle=True):
        self.df = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return len(self.df) // self.batch_size

    def __getitem__(self, index):
        batch = self.df.iloc[
            index * self.batch_size:(index+1) * self.batch_size
        ]
        images, labels = [], []
        for _, row in batch.iterrows():
            path = get_image_path(row['Image Index'])
            if path is None:
                continue
            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224, 224))
            img = img / 255.0
            images.append(img)
            labels.append(row['encoded_labels'])
        return np.array(images), np.array(labels)

    def on_epoch_end(self):
        if self.shuffle:
            self.df = self.df.sample(frac=1).reset_index(drop=True)

# Class weights
class_weights = {}
for i, disease in enumerate(DISEASES):
    positive = train_df['encoded_labels'].apply(lambda x: x[i]).sum()
    negative = len(train_df) - positive
    class_weights[i] = negative / (positive + 1e-6)

# Generators
train_gen = XRayGenerator(train_df, batch_size=128, shuffle=True)
test_gen  = XRayGenerator(test_df,  batch_size=128, shuffle=False)

print("Generators ready!")
print("Train batches:", len(train_gen))
print("Test batches:", len(test_gen))

In [ ]:
base_model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output = layers.Dense(14, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

print("Model ready!")
print("Total layers:", len(model.layers))

In [ ]:
MODEL_PATH = '/kaggle/working/best_model2.keras'

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc', patience=3,
        factor=0.5, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_PATH,
        save_best_only=True,
        monitor='val_auc',
        mode='max', verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        patience=10, monitor='val_auc',
        mode='max', verbose=1
    )
]

print("Callbacks ready!")

In [ ]:
history = model.fit(
    train_gen,
    epochs=15,
    initial_epoch=0,
    validation_data=test_gen,
    callbacks=callbacks,
    class_weight=class_weights
)

print("Training complete!")

In [ ]:
model.save('/kaggle/working/best_model2.keras')

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False
    
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

# Fine tuning callbacks
ft_callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc', patience=3,
        factor=0.5, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        '/kaggle/working/best_model_finetuned.keras',
        save_best_only=True,
        monitor='val_auc',
        mode='max', verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        patience=7, monitor='val_auc',
        mode='max', verbose=1
    )
]

# Fine tuning training
history_ft = model.fit(
    train_gen,
    epochs=10,
    validation_data=test_gen,
    callbacks=ft_callbacks,
    class_weight=class_weights
)

print("Fine tuning complete!")

In [ ]:
if os.path.exists(MODEL_PATH):
    size = os.path.getsize(MODEL_PATH) / (1024*1024)
    print(f"Model saved! Size: {size:.2f} MB")
else:
    print("Model not found!")

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

def generate_gradcam(model, img_array, class_index):
    # Last conv layer
    last_conv_layer = model.get_layer('conv5_block16_2_conv')
    
    grad_model = tf.keras.Model(
        inputs=model.input,
        outputs=[last_conv_layer.output, model.output]
    )
    
    with tf.GradientTape() as tape:
        conv_output, predictions = grad_model(img_array)
        loss = predictions[:, class_index]
    
    grads = tape.gradient(loss, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_output = conv_output[0]
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    heatmap = heatmap.numpy()
    
    # Heatmap resizing
    heatmap = cv2.resize(heatmap, (224, 224))
    heatmap = np.uint8(255 * heatmap)
    heatmap_colored = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    
    original = np.uint8(img_array[0] * 255)
    overlaid = cv2.addWeighted(original, 0.6, heatmap_colored, 0.4, 0)
    
    return overlaid, heatmap

print("Grad-CAM function ready!")

In [ ]:
model.save('/kaggle/working/best_model.keras')
print("Size:", os.path.getsize('/kaggle/working/best_model.keras') / (1024*1024), "MB")
print("Done!")

In [ ]:
df[df['Finding Labels'].str.contains('Fibrosis')]['Image Index'].tail(10)

In [ ]:
import os
target=" 00001833_008.png"
for root,dirs,files in os.walk("/kaggle/input"):
    if target in files:
        print(os.path.join(root,target))

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

target = "00001833_008.png"

BASE_PATH = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"

folders = [
    "images_001","images_002","images_003","images_004",
    "images_005","images_006","images_007","images_008",
    "images_009","images_010","images_011","images_012"
]

path = None

for folder in folders:
    temp_path = f"{BASE_PATH}/{folder}/images/{target}"
    if os.path.exists(temp_path):
        path = temp_path
        break

if path is None:
    print("Image not found")
else:
    print("Found at:", path)
    
    img = Image.open(path)
    plt.imshow(img)
    plt.axis("off")

In [ ]:
import shutil

src = path   # jo path mila tha search se
dst = "/kaggle/working/00001833_008.png"

shutil.copy(src, dst)

print("Saved at:", dst)

In [ ]:
img_name = "00001833_008.png"

img_path = get_image_path(img_name)

print("Image path:", img_path)

In [ ]:
def preprocess_image(img_path):
    img = cv2.imread(img_path)
    
    if img is None:
        raise ValueError("❌ Image not loaded, Wrong path!")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)
    return img

img = preprocess_image(img_path)

In [ ]:
predictions = model.predict(img)[0]

In [ ]:
results = []

for disease, prob in zip(DISEASES, predictions):
    results.append({
        "disease": disease,
        "probability": float(prob) * 100,
        "detected": prob > 0.5
    })

results = sorted(results, key=lambda x: x["probability"], reverse=True)

print("\n DIAGNOSIS REPORT")
print("━━━━━━━━━━━━━━━━━━━━━━")

for r in results:
    status = "🔴 DETECTED" if r["detected"] else "🟢"
    print(f"{r['disease']:20} → {r['probability']:.2f}% {status}")